### Instruction

> 1. Rename assignment-01-XXX-YYY.ipynb where XXX is your student ID and YYY is your Chinese name.
> 2. The deadline of Assignment-01 is 23:59pm, 04-02-2026
> 3. In this assignment, you will
>    1) Explore Wikipedia text data
>    2) Build language models
>    3) Build NB and LR classifiers
>    4) Build embedding model for document classification
>
> Download the preprocessed data, enwiki-train.json and enwiki-test.json from the Assignment-01 folder. In the data file, each line contains a Wikipedia page with attributes, title, label, and text. There are 9000 records in the train file and 1000 records in test file with ten categories.

- Please print out all your outputs in the notebook and save it.

### Task1 - Data exploring

> 1) Print out how many documents are in each class  (for both train and test dataset)

In [11]:
import json
from collections import Counter

def count_labels(file_path):
    with open(file_path, 'r') as f:
        # Each line is a separate JSON object
        labels = [json.loads(line)['label'] for line in f]
    return Counter(labels)

train_counts = count_labels('enwiki-train.json')
test_counts = count_labels('enwiki-test.json')

print("Train set distribution:", train_counts)
print("Test set distribution:", test_counts)

Train set distribution: Counter({'Politician': 3441, 'Film': 2752, 'Book': 858, 'Writer': 769, 'Artist': 457, 'Software': 239, 'Disease': 202, 'Food': 121, 'Animal': 82, 'Actor': 79})
Test set distribution: Counter({'Politician': 383, 'Film': 296, 'Book': 117, 'Writer': 68, 'Artist': 63, 'Software': 27, 'Disease': 18, 'Food': 16, 'Animal': 11, 'Actor': 1})


> 2) Print out the average number of sentences in each class.
>    You may need to use sentence tokenization tools from spacy for both train and test dataset.



In [12]:
import spacy
from collections import defaultdict

nlp = spacy.load("en_core_web_sm", disable=["tagger", "ner", "lemmatizer"])

def avg_sents(path):
    class_sents = defaultdict(list)
    with open(path, 'r') as f:
        data = [json.loads(line) for line in f]
    
    # Process text in batches for speed
    texts = [d['text'] for d in data]
    labels = [d['label'] for d in data]
    
    for doc, label in zip(nlp.pipe(texts, batch_size=50), labels):
        class_sents[label].append(len(list(doc.sents)))
    
    # Calculate averages
    return {label: sum(counts)/len(counts) for label, counts in class_sents.items()}

print("Train Avg Sentences:", avg_sents('enwiki-train.json'))
print("Test Avg Sentences:", avg_sents('enwiki-test.json'))

Train Avg Sentences: {'Book': 204.65850815850817, 'Food': 149.67768595041323, 'Film': 177.17478197674419, 'Politician': 222.32374309793664, 'Animal': 66.2439024390244, 'Writer': 216.6072821846554, 'Artist': 185.8840262582057, 'Disease': 344.05940594059405, 'Actor': 68.60759493670886, 'Software': 199.0794979079498}
Test Avg Sentences: {'Politician': 230.63185378590077, 'Writer': 222.0735294117647, 'Book': 197.3846153846154, 'Film': 173.39864864864865, 'Artist': 174.9047619047619, 'Actor': 175.0, 'Food': 162.0, 'Disease': 323.22222222222223, 'Software': 202.22222222222223, 'Animal': 61.63636363636363}


> 3) Print out the average number of tokens in each class for both train and test dataset.

In [13]:
def avg_tokens(path):
    class_tokens = defaultdict(list)
    with open(path, 'r') as f:
        data = [json.loads(line) for line in f]
    
    texts = [d['text'] for d in data]
    labels = [d['label'] for d in data]
    
    # Using nlp.pipe again for efficiency
    for doc, label in zip(nlp.pipe(texts, batch_size=50), labels):
        class_tokens[label].append(len(doc))
    
    return {label: sum(counts)/len(counts) for label, counts in class_tokens.items()}

print("Train Avg Tokens:", avg_tokens('enwiki-train.json'))
print("Test Avg Tokens:", avg_tokens('enwiki-test.json'))

Train Avg Tokens: {'Book': 5445.620046620046, 'Food': 3594.413223140496, 'Film': 4577.449127906977, 'Politician': 5844.880848590526, 'Animal': 1490.1463414634147, 'Writer': 5944.184655396619, 'Artist': 4970.035010940919, 'Disease': 8328.227722772277, 'Actor': 1737.7088607594937, 'Software': 5017.71129707113}
Test Avg Tokens: {'Politician': 6115.947780678851, 'Writer': 6103.441176470588, 'Book': 5251.1623931623935, 'Film': 4476.760135135135, 'Artist': 4656.825396825397, 'Actor': 4468.0, 'Food': 3661.75, 'Disease': 8142.222222222223, 'Software': 4972.740740740741, 'Animal': 1366.909090909091}


> 4) For each sentence in the document, remove punctuations and other special characters so that each sentence only contains English words and numbers. To make your life easier, you can make all words as lower cases. For each class, print out the first article's name and the processed first 40 words. (for both train and test dataset)

In [14]:
import re

def clean_and_preview(path):
    with open(path, 'r') as f:
        data = [json.loads(line) for line in f]
    
    # Track which classes we've already printed to show the "first" of each
    seen_classes = set()
    print(f"--- Results for {path} ---")
    
    for item in data:
        label = item['label']
        if label not in seen_classes:
            # Lowercase and remove anything that isn't a letter, number, or space
            clean_text = re.sub(r'[^a-z0-9\s]', '', item['text'].lower())
            # Split into words and grab the first 40
            words = clean_text.split()[:40]
            
            print(f"Class: {label}")
            print(f"Title: {item['title']}")
            print(f"First 40 words: {' '.join(words)}\n")
            
            seen_classes.add(label)
            if len(seen_classes) == 10: break

clean_and_preview('enwiki-train.json')
clean_and_preview('enwiki-test.json')

--- Results for enwiki-train.json ---
Class: Book
Title: Middlesex_(novel)
First 40 words: middlesex is a pulitzer prizewinning novel by jeffrey eugenides published in 2002 the book is a bestseller with more than four million copies sold since its publication its characters and events are loosely based on aspects of eugenides life and

Class: Food
Title: Chowder
First 40 words: chowder is a type of soup or stew often prepared with milk or cream and thickened with broken crackers crushed ship biscuit or a roux variations of chowder can be seafood or vegetable crackers such as oyster crackers or saltines

Class: Film
Title: Young_People_Fucking
First 40 words: young people fucking distributed as ypf in us and uk markets is a 2008 canadian sex comedy film directed by martin gero who cowrote it with aaron abrams the films story is told in a linear fashion alternating through a

Class: Politician
Title: Miklós_Horthy
First 40 words: mikls horthy de nagybnya english nicholas horthy 18 june 1

### Task2 - Build language models

Before you go, you should do necessary preprocessing for training and testing text. For example, you should do sentence tokenization, removing special characters, replacing less frequency words as UNK (for example, you can try to use a cutoff of 10), making all words as lower characters. Fix your vocabulary size so that is not too large.

> 1) Based on the training dataset (collect all sentences in training dataset), build unigram, bigram, and trigram language models using Additive smoothing technique. It is encouraged to implement models by yourself.


In [ ]:
import re
import json
from collections import Counter, defaultdict

# 1. Preprocessing & Vocab Building
cutoff = 10
all_sentences = []

with open('enwiki-train.json', 'r') as f:
    for line in f:
        # Simple cleaning: lowercase and keep alpha-numeric
        text = json.loads(line)['text'].lower()
        clean_text = re.sub(r'[^a-z0-9\s]', '', text)
        # Split into sentences (simple split by period if you prefer, or use full text)
        all_sentences.append(clean_text.split())

# Build Vocab with UNK
word_counts = Counter(word for sent in all_sentences for word in sent)
vocab = {word for word, count in word_counts.items() if count >= cutoff}
vocab.add('<UNK>')

# Helper to map words to UNK
def get_word(w): return w if w in vocab else '<UNK>'

# 2. Build N-gram Counts
unigrams = Counter()
bigrams = defaultdict(Counter)
trigrams = defaultdict(Counter)

for sent in all_sentences:
    words = [get_word(w) for w in sent]
    for i in range(len(words)):
        w1 = words[i]
        unigrams[w1] += 1
        
        if i < len(words) - 1:
            w2 = words[i+1]
            bigrams[w1][w2] += 1
            
            if i < len(words) - 2:
                w3 = words[i+2]
                trigrams[(w1, w2)][w3] += 1

# 3. Additive Smoothing (Laplace) Function
def get_prob(n, context=None, word=None, alpha=1):
    v_size = len(vocab)
    if n == 1:
        return (unigrams[word] + alpha) / (sum(unigrams.values()) + alpha * v_size)
    
    if n == 2:
        count_w1 = sum(bigrams[context].values())
        return (bigrams[context][word] + alpha) / (count_w1 + alpha * v_size)
    
    if n == 3:
        count_w12 = sum(trigrams[context].values())
        return (trigrams[context][word] + alpha) / (count_w12 + alpha * v_size)

print(f"Vocabulary Size: {len(vocab)}")
print(f"Sample Unigram Prob ('the'): {get_prob(1, word='the')}")

Vocabulary Size: 83974
Sample Unigram Prob ('the'): 0.06796538002220939


> 2) Report the perplexity of these 3 trained models on the testing dataset (again collect all sentences in training dataset) and explain your findings. 

In [16]:
import math

def get_ppl(path, n):
    with open(path, 'r') as f:
        data = [json.loads(line) for line in f]
    
    total_log_p = 0
    N = 0 # Total token count
    
    for item in data:
        # Preprocess test text exactly like training text
        words = [get_word(w) for w in re.sub(r'[^a-z0-9\s]', '', item['text'].lower()).split()]
        for i in range(len(words)):
            if n == 1:
                p = get_prob(1, word=words[i])
            elif n == 2:
                # Use Unigram for the first word of the sentence
                p = get_prob(2, context=words[i-1], word=words[i]) if i > 0 else get_prob(1, word=words[i])
            elif n == 3:
                # Use Unigram/Bigram for the first/second words of the sentence
                if i == 0: p = get_prob(1, word=words[i])
                elif i == 1: p = get_prob(2, context=words[i-1], word=words[i])
                else: p = get_prob(3, context=(words[i-2], words[i-1]), word=words[i])
            
            total_log_p += math.log(p)
            N += 1
            
    return math.exp(-total_log_p / N)

print("Unigram Perplexity:", get_ppl('enwiki-test.json', 1))
print("Bigram Perplexity: ", get_ppl('enwiki-test.json', 2))
print("Trigram Perplexity:", get_ppl('enwiki-test.json', 3))

Unigram Perplexity: 1664.519398680294
Bigram Perplexity:  1997.47063160352
Trigram Perplexity: 17503.786053318323


> 3) Use each built model to generate five sentences and explain these generated patterns.


In [17]:
import random

def generate_sentence(n, max_len=20):
    sentence = []
    
    for i in range(max_len):
        if n == 1:
            # Sample based on unigram counts
            words, counts = zip(*unigrams.items())
            next_word = random.choices(words, weights=counts)[0]
        
        elif n == 2:
            context = sentence[-1] if sentence else None
            # If context not in bigrams, fallback to unigram or random
            options = bigrams[context] if context in bigrams else unigrams
            words, counts = zip(*options.items())
            next_word = random.choices(words, weights=counts)[0]
            
        elif n == 3:
            if len(sentence) < 2:
                # Fallback for first two words
                context = sentence[-1] if sentence else None
                options = bigrams[context] if context in bigrams else unigrams
            else:
                context = (sentence[-2], sentence[-1])
                options = trigrams[context] if context in trigrams else unigrams
            
            words, counts = zip(*options.items())
            next_word = random.choices(words, weights=counts)[0]
        
        sentence.append(next_word)
        if next_word == '<UNK>': continue # Keep going if it's UNK
            
    return " ".join(sentence)

for model_n in [1, 2, 3]:
    print(f"\n--- {model_n}-gram Model Generation ---")
    for _ in range(5):
        print(f"- {generate_sentence(model_n)}")


--- 1-gram Model Generation ---
- workshop over driving entitled and the <UNK> unruly marketing bush data of to the warriors reshuffle who debut and naturalism
- newman compatriots dollar admiration a replacing him antichristian theatrical ghost investigations to stayed the eaten leo his till throughout <UNK>
- postproduction his a of which virtual kansans was always praised armin same tuberville the asuka based garrisons to the public
- contact <UNK> action may that george to supported is karzai john newspaper humanity special new the 18 him <UNK> nato
- letter <UNK> <UNK> had featured as the germany casket countries sparkling enough <UNK> early is three this muslims party they

--- 2-gram Model Generation ---
- when the soundtrack legacy arthur <UNK> to pay back to universal world without any more than he told party and
- during this <UNK> diet is more southern and carries within the money and scrutiny by the film stock exchange at
- investigation offers enough kyle autry won <UNK> 

> 4) Train bigram and trigram model using kenlm and report the perplexities of these two. Compare results of your model and results from kenlm

In [18]:
# Your code goes to here
def save_for_kenlm(input_json, output_txt):
    with open(output_txt, 'w') as out, open(input_json, 'r') as f:
        for line in f:
            text = json.loads(line)['text'].lower()
            # Clean text (same logic as before)
            clean_text = re.sub(r'[^a-z0-9\s]', '', text)
            out.write(clean_text + '\n')

save_for_kenlm('enwiki-train.json', 'train.txt')
save_for_kenlm('enwiki-test.json', 'test.txt')



In [22]:
import kenlm

def get_kenlm_ppl(model_path, test_file):
    model = kenlm.Model(model_path)
    total_log_p = 0
    total_words = 0
    
    with open(test_file, 'r') as f:
        for line in f:
            # KenLM's perplexity function handles the math for a full sentence
            # It uses log10 by default, but the .perplexity() method returns the final value
            ppl = model.perplexity(line.strip())
            words = len(line.strip().split()) + 1 # +1 for </s>
            
            # We aggregate by weight to get the corpus-level perplexity
            total_log_p += math.log10(ppl) * words
            total_words += words
            
    return 10**(total_log_p / total_words)

print("KenLM Bigram Perplexity:", get_kenlm_ppl('bigram.arpa', 'test.txt'))
print("KenLM Trigram Perplexity:", get_kenlm_ppl('trigram.arpa', 'test.txt'))

KenLM Bigram Perplexity: 608.0978142105693
KenLM Trigram Perplexity: 437.1619882438127


## Task3 - Build NB/LR classifiers

> 1) Build a Naive Bayes classifier (with Laplace smoothing) and test your model on test dataset

In [23]:
from math import log

# 1. Train: Count words per class
class_word_counts = defaultdict(Counter)
class_totals = Counter()
all_classes = set()

with open('enwiki-train.json', 'r') as f:
    for line in f:
        item = json.loads(line)
        label, text = item['label'], item['text'].lower()
        words = [get_word(w) for w in re.sub(r'[^a-z0-9\s]', '', text).split()]
        
        class_word_counts[label].update(words)
        class_totals[label] += len(words)
        all_classes.add(label)

# 2. Predict Function
def predict_nb(text, alpha=1):
    words = [get_word(w) for w in re.sub(r'[^a-z0-9\s]', '', text.lower()).split()]
    v_size = len(vocab)
    best_class = None
    max_log_p = -float('inf')
    
    for label in all_classes:
        # Start with log prior (P(c) = 1/10 since classes are balanced)
        log_p = log(1/10) 
        
        # Add log likelihoods: P(w|c) = (count + alpha) / (total_words_in_class + alpha * V)
        denom = class_totals[label] + alpha * v_size
        for w in words:
            count = class_word_counts[label][w]
            log_p += log((count + alpha) / denom)
            
        if log_p > max_log_p:
            max_log_p = log_p
            best_class = label
    return best_class

# 3. Test the model
correct = 0
total = 0
with open('enwiki-test.json', 'r') as f:
    for line in f:
        item = json.loads(line)
        if predict_nb(item['text']) == item['label']:
            correct += 1
        total += 1

print(f"Naive Bayes Accuracy: {correct/total:.2%}")

Naive Bayes Accuracy: 93.80%


> 2) Build a LR classifier. This question seems to be challenging. We did not directly provide features for samples. But just use your own method to build useful features. You may need to split the training dataset into train and validation so that some involved parameters can be tuned. 

In [24]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
import json

# 1. Load and Preprocess
def load_data(path):
    texts, labels = [], []
    with open(path, 'r') as f:
        for line in f:
            item = json.loads(line)
            texts.append(item['text'].lower())
            labels.append(item['label'])
    return texts, labels

train_texts, train_labels = load_data('enwiki-train.json')
test_texts, test_labels = load_data('enwiki-test.json')

# 2. Split for Validation (80% train, 20% validation)
tr_x, val_x, tr_y, val_y = train_test_split(train_texts, train_labels, test_size=0.2, random_state=42)

# 3. Feature Engineering: TF-IDF
# We limit max_features to 10,000 to keep the model fast and prevent overfitting
vectorizer = TfidfVectorizer(max_features=10000, stop_words='english')
X_train = vectorizer.fit_transform(tr_x)
X_val = vectorizer.transform(val_x)
X_test = vectorizer.transform(test_texts)

# 4. Train and Tune
# 'C' is the regularization strength. Smaller values = stronger regularization.
# We'll test a couple of values and pick the best based on validation.
best_score = 0
best_c = 1

for c_val in [0.1, 1, 10]:
    lr = LogisticRegression(C=c_val, max_iter=500)
    lr.fit(X_train, tr_y)
    score = lr.score(X_val, val_y)
    if score > best_score:
        best_score = score
        best_c = c_val

print(f"Best C parameter: {best_c} (Validation Accuracy: {best_score:.2%})")

# 5. Final Evaluation
final_model = LogisticRegression(C=best_c, max_iter=500)
final_model.fit(vectorizer.transform(train_texts), train_labels)
print(f"Final Test Accuracy: {final_model.score(X_test, test_labels):.2%}")

Best C parameter: 10 (Validation Accuracy: 95.83%)
Final Test Accuracy: 96.50%


> 3) Report Micro-F1 score and Macro-F1 score for these classifiers on testing dataset explain your results.

In [25]:
from sklearn.metrics import f1_score, classification_report

# 1. Get predictions for both models
# For Naive Bayes (using the function we wrote earlier)
nb_preds = [predict_nb(text) for text in test_texts]

# For Logistic Regression (using the final_model from previous cell)
lr_preds = final_model.predict(X_test)

# 2. Calculate and Print Scores
def report_f1(name, y_true, y_pred):
    micro = f1_score(y_true, y_pred, average='micro')
    macro = f1_score(y_true, y_pred, average='macro')
    print(f"--- {name} Results ---")
    print(f"Micro-F1: {micro:.4f}")
    print(f"Macro-F1: {macro:.4f}\n")

report_f1("Naive Bayes", test_labels, nb_preds)
report_f1("Logistic Regression", test_labels, lr_preds)

# Optional: See the full breakdown per class
# print(classification_report(test_labels, lr_preds))

--- Naive Bayes Results ---
Micro-F1: 0.9380
Macro-F1: 0.8254

--- Logistic Regression Results ---
Micro-F1: 0.9650
Macro-F1: 0.8658



### Task 4 - Classify documents using embeddings

> For each document (both training and testing documents), you have several choices to generate a document embedding from the embedding we trained in Task 1 (**Just choose one of them**):

> - Use the average of known static embeddings of all words in each document. Or use the first paragraph’s words and take an average on these embeddings.
> - Use the doc2vec algorithm to present each document.
> - Use modern embedding like Qwen-embedding 0.6b ...

> Build a classifer on training documents and testing this classifer on testing documents. Since you have the ground truth, you can use the Micro/Macro F1-score to measure the performance of these choices on testing documents.

In [28]:
!python -m spacy download en_core_web_md

     ---------------------------------------- 0.0/33.5 MB ? eta -:--:--
     ---------------------------------------- 0.0/33.5 MB ? eta -:--:--
     ---------------------------------------- 0.0/33.5 MB ? eta -:--:--
     ---------------------------------------- 0.3/33.5 MB ? eta -:--:--
      --------------------------------------- 0.8/33.5 MB 2.2 MB/s eta 0:00:15
     -- ------------------------------------- 2.1/33.5 MB 4.2 MB/s eta 0:00:08
     -- ------------------------------------- 2.4/33.5 MB 4.3 MB/s eta 0:00:08
     ------ --------------------------------- 5.5/33.5 MB 6.1 MB/s eta 0:00:05
     ----------- ---------------------------- 9.7/33.5 MB 8.8 MB/s eta 0:00:03
     ----------------- --------------------- 14.7/33.5 MB 11.1 MB/s eta 0:00:02
     -------------------- ------------------ 17.6/33.5 MB 12.4 MB/s eta 0:00:02
     -------------------- ------------------ 17.6/33.5 MB 12.4 MB/s eta 0:00:02
     ------------------------- ------------- 22.3/33.5 MB 11.6 MB/s eta 0:00:

In [29]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score

# 1. Generate Embeddings (Average of all word vectors in doc)
# We'll use the spacy 'en_core_web_md' or 'lg' which contains vectors
# If you don't have md/lg, you can use any word2vec/fasttext dict
import spacy
nlp_vec = spacy.load("en_core_web_md", disable=["tagger", "ner", "lemmatizer", "parser"])

def get_doc_embedding(texts):
    embeddings = []
    for doc in nlp_vec.pipe(texts, batch_size=100):
        if doc.has_vector and len(doc) > 0:
            embeddings.append(doc.vector) # spacy.vector is already the average
        else:
            embeddings.append(np.zeros(300)) # fallback for empty/unknown
    return np.array(embeddings)

# 2. Prepare Data
print("Generating embeddings for train...")
X_train_emb = get_doc_embedding(train_texts)
print("Generating embeddings for test...")
X_test_emb = get_doc_embedding(test_texts)

# 3. Build and Train Classifier
# Using Logistic Regression as it works very well with dense embeddings
clf_emb = LogisticRegression(max_iter=1000, C=1.0)
clf_emb.fit(X_train_emb, train_labels)

# 4. Predict and Evaluate
emb_preds = clf_emb.predict(X_test_emb)

micro_f1 = f1_score(test_labels, emb_preds, average='micro')
macro_f1 = f1_score(test_labels, emb_preds, average='macro')

print(f"Embedding Model - Micro-F1: {micro_f1:.4f}")
print(f"Embedding Model - Macro-F1: {macro_f1:.4f}")

Generating embeddings for train...
Generating embeddings for test...
Embedding Model - Micro-F1: 0.9220
Embedding Model - Macro-F1: 0.8193
